# DINOv2 · pipeline de adaptación por componentes
**Estado: estudio completo preparado, sin resultados hasta ejecutar/publicar.** Objetivo: medir el aporte de proyección, LoRA y su combinación sobre DINO congelado. Pregunta: ¿mejora cada intervención la recuperación de identidades? Hipótesis: puede mejorar, pero el control sin ajuste sigue siendo elegible. Tres semillas por familia; mismo corpus, detector, procesador, evaluación y presupuesto máximo de pasos. La proyección usa features congeladas cacheadas; LoRA cambia el encoder. La combinación ajusta simultáneamente proyección y LoRA desde C1: no equivale al E4 histórico de CLIP. No se atribuye novedad a métodos existentes.
[DINOv2](https://arxiv.org/abs/2304.07193), [LoRA](https://arxiv.org/abs/2106.09685), [batch-hard triplet](https://arxiv.org/abs/1703.07737). Selección, cabeza identidad y controles son decisiones de este estudio. El test ya motivó investigar DINO; estos resultados serán retrospectivos y no confirmación independiente.
## Fase 1. Preparación y verificación de datos
Cargar el corpus preparado v2: 193.276 train y 273 val. Test público v1 inmutable: 277 fotos; PetTrace v6: 502, siempre separado. Las cajas v2 no cambian el test. Sin limpieza en el notebook. Runtime CUDA y código local del repositorio instalado antes de abrir el notebook. Credenciales desde entorno o Colab Secrets, nunca literales.

In [ ]:
import sys, os, json, gc, shutil, time
from pathlib import Path
from dataclasses import asdict
if not (3,11) <= sys.version_info[:2] < (3,14):
    raise RuntimeError('Python 3.11–3.13 required')
import numpy as np
import torch
import mlflow
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.verification import sha256
if not torch.cuda.is_available():raise RuntimeError('The full study requires CUDA')
for name in ('HF_TOKEN','MLFLOW_TRACKING_USERNAME','MLFLOW_TRACKING_PASSWORD'):
    if not os.environ.get(name):
        from google.colab import userdata
        os.environ[name]=userdata.get(name)
    if not os.environ[name]:raise RuntimeError('Missing '+name)
os.environ['MLFLOW_TRACKING_URI']=PRODUCTION_URI
os.environ['MLFLOW_REGISTRY_URI']=PRODUCTION_URI
mlflow.set_tracking_uri(PRODUCTION_URI);mlflow.set_registry_uri(PRODUCTION_URI)
LOG_TO_MLFLOW=True
REGISTER_MODEL=True
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
SEEDS=(45,46,47)
MODES=('projection','lora','combined')
REPORT=Path('reports/dino-components/20261118-full-v1')
REPORT.mkdir(parents=True,exist_ok=True)
SOURCE=source_identity()
torch.set_num_threads(4)
from pettrace.data import training
from pettrace.experiments.e3 import load_benchmarks
from pettrace.evaluation.core import load_model
from pettrace.training.dinov2 import build, Config, Pipeline, Preprocessor, fit, REVISION, REPOSITORY
from pettrace.training import dino_metric, lora
from pettrace.experiments.dinov2 import frozen_features, select_family, publish_training
prepared=training(version='v2',cache_dir=str(REPORT/'data'))
assert prepared.fingerprint=='d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58'
rows=[r for r in prepared.rows if r['split'] in ('train','val')]
assert sum(r['split']=='train' for r in rows)==193276
assert sum(r['split']=='val' for r in rows)==273
benchmarks=load_benchmarks(prepared,REPORT,private_version='v6',private_commit=DATASET_COMMIT)
base=load_model('models:/pettrace-embedder/2');assert base.golden_ok
crop=lora.PreparedPreprocessor(base.pipeline,prepared)
gate=[next(r for r in rows if r['split']=='train' and r['species']==sp) for sp in ('cat','dog')]
crop.validate(gate)
study=dict(source=SOURCE,model_revision=REVISION,prepared=prepared.provenance,seeds=SEEDS,modes=MODES,private_commit=DATASET_COMMIT,register_model=REGISTER_MODEL,selection='family mean val mAP with macro Rank1 guard; frozen eligible; then seed by val mAP',training_budget='30 epochs x 100 steps maximum; patience 7; P8K4; different actual steps and GPU time',projection_lr=1e-4,lora_lr=1e-5,combined_lr=1e-5)
# Reject reuse after code/configuration changes; use a new study directory instead.
study=json.loads(json.dumps(study))
if (REPORT/'study.json').exists():assert json.loads((REPORT/'study.json').read_text())==study
else:(REPORT/'study.json').write_text(json.dumps(study,indent=2))
print('Full data verified. Study:',REPORT)


## Fase 2. Representación y entrenamiento
CLS 1024 normalizado y procesador oficial 256→224. Proyección identidad 1024×1024; LoRA query/value rango 8, alpha 16. Tres semillas por familia, P8×K4, margen 0,2, AdamW, clipping 5. Proyección lr 1e-4; LoRA y combinación lr 1e-5. Máximo 30×100 pasos, paciencia 7. Selección de época por mAP macro de especie, incluida época cero. Parámetros/costes difieren: no es una comparación a igual capacidad ni tiempo GPU. Se publica cada entrenamiento antes de evaluar tests. La selección permite conservar C1.

In [ ]:
from pettrace.evaluation.metrics import retrieval
from pettrace.models.golden import golden_images, compare, fingerprint
from huggingface_hub import snapshot_download
model_dir=Path(snapshot_download(REPOSITORY,revision=REVISION,allow_patterns=['model.safetensors','config.json','preprocessor_config.json']))
base_hashes={f:sha256(model_dir/f) for f in ('model.safetensors','config.json','preprocessor_config.json')}
base_hashes.update({'detector.'+k:v for k,v in base.sha256.items() if 'detector' in k})
encoder,processor=build('frozen','cuda')
pipeline=Pipeline(encoder,processor,base.pipeline.detector)
control_golden=pipeline.embed(golden_images());np.save(REPORT/'control-golden.npy',control_golden)
x=frozen_features(pipeline,rows,REPORT/'features',dict(model=base_hashes,processor=processor.to_dict(),source=SOURCE,detector_spec=base.pipeline.spec))
vi=[i for i,r in enumerate(rows) if r['split']=='val'];val=[rows[i] for i in vi]
control_metrics=retrieval(x[vi],[r['animal'] for r in val],[r['species'] for r in val])
control={k:float(np.mean([control_metrics[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
(REPORT/'control-validation.json').write_text(json.dumps(control_metrics,indent=2))
del encoder,pipeline;gc.collect();torch.cuda.empty_cache()
candidates=[]
for mode in MODES:
    for seed in SEEDS:
        folder=REPORT/f'{mode}-{seed}';folder.mkdir(exist_ok=True)
        config=Config(seed=seed,learning_rate=1e-4 if mode=='projection' else 1e-5)
        encoder,processor=build(mode,'cuda',config)
        pipeline=Pipeline(encoder,processor,base.pipeline.detector)
        assert compare(control_golden,pipeline.embed(golden_images()))[0]
        started=time.monotonic();torch.cuda.reset_peak_memory_stats()
        was_complete=(folder/'result.json').exists() or (folder/'training-result.json').exists()
        if mode=='projection':
            if (folder/'result.json').exists():result=json.loads((folder/'result.json').read_text())
            else:_,result=dino_metric.fit(x,rows,folder,seed)
            encoder.head.load_state_dict(torch.load(folder/'metric.pt',weights_only=True,map_location='cpu'))
            checkpoint=folder/'metric.pt';metrics=result['best']
        else:
            if (folder/'training-result.json').exists():
                result=json.loads((folder/'training-result.json').read_text());lora.load_adapter(encoder,folder/'adapter.pt')
            else:encoder,result=fit(encoder,rows,Preprocessor(processor,base.pipeline.detector,crop),folder,config,'cuda')
            checkpoint=folder/'adapter.pt'
            _,values=lora._validation_score(encoder,val,Preprocessor(processor,base.pipeline.detector,crop).pixels(val),'cuda')
            metrics={k:float(np.mean([values[sp][k] for sp in ('cat','dog')])) for k in ('mAP','rank1')}
            (folder/'validation.json').write_text(json.dumps(values,indent=2))
        if not was_complete:
            (folder/'cost.json').write_text(json.dumps(dict(training_and_validation_seconds=time.monotonic()-started,peak_cuda_bytes=torch.cuda.max_memory_allocated(),training_backend='cpu' if mode=='projection' else 'cuda'),indent=2))
        design=dict(study=study,mode=mode,seed=seed,config=asdict(config),checkpoint_sha256=sha256(checkpoint),trainable_parameters=lora.trainable_parameters(encoder),base_hashes=base_hashes)
        training_run=publish_training(folder,design,metrics)
        candidates.append(dict(mode=mode,seed=seed,validation=metrics,folder=str(folder),training_run=training_run))
        del pipeline,encoder;gc.collect();torch.cuda.empty_cache()
selection=select_family(candidates,control)
(REPORT/'selection.json').write_text(json.dumps(selection,indent=2))
(REPORT/'candidates.json').write_text(json.dumps(candidates,indent=2))
print('Selection frozen before test:',selection)


## Fase 3. Evaluación y análisis
Evaluar las nueve corridas y el control congelado sobre los dos dominios por separado. Galería cerrada de misma especie, leave-one-out por fotografía, Rank-k por identidad (máximo), mAP por fotografías; EER descriptiva, no umbral operativo. Publicar negativos, incertidumbre y dispersión entre semillas; no seleccionar por test. El registro opcional conserva su configuración y usa un registro de investigación separado, sin promoción de aliases. El bundle PyTorch requiere el código adjunto y los pesos HF fijados: no es todavía el artefacto ONNX servido por la app. Continúan pendientes captura entre sesiones, coste comparado con CLIP y confirmación independiente.

In [ ]:
from pettrace.models.artifacts import Model
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.experiments.zero_shot import save_publication_input,load_publication_input
client=mlflow.MlflowClient();publications=[]
for candidate in [dict(mode='frozen',seed=45,folder=str(REPORT/'frozen'))]+candidates:
    mode,seed=candidate['mode'],candidate['seed'];folder=Path(candidate['folder']);folder.mkdir(exist_ok=True)
    encoder,processor=build(mode,'cuda',Config(seed=seed))
    hashes=dict(base_hashes)
    if mode=='projection':
        encoder.head.load_state_dict(torch.load(folder/'metric.pt',weights_only=True,map_location='cpu'));hashes['metric.pt']=sha256(folder/'metric.pt')
    elif mode!='frozen':
        lora.load_adapter(encoder,folder/'adapter.pt');hashes['adapter.pt']=sha256(folder/'adapter.pt')
    pipeline=Pipeline(encoder,processor,base.pipeline.detector)
    golden=pipeline.embed(golden_images());assert compare(golden,pipeline.embed(golden_images()))[0]
    np.save(folder/'golden.npy',golden)
    model=Model(f'pettrace-dino://{REPORT.name}/{mode}/{seed}',str(folder),pipeline,f'DINO-{mode}-{seed}','',hashes,True,fingerprint(golden))
    for domain,ds in (('public',benchmarks.public),('private',benchmarks.private)):
        target=folder/domain;target.mkdir(exist_ok=True)
        if (target/'evaluation-input.json').exists():
            ev=load_publication_input(target);ev.dataset=ds
        else:
            ev=evaluate(ds,pipeline,zero_shot=True);save(ev,model,str(target));save_publication_input(ev,target)
            (target/'runtime.json').write_text(json.dumps(dict(**SOURCE,execution_id=REPORT.name,backend='torch_cuda',model_sha256=hashes),indent=2))
        run=publish_evaluation(ev,model,target,SOURCE['source_sha256'],experiment='DINO-'+mode,expected_model_uri=model.uri,expected_model_sha256=hashes,expected_dataset_version=ds.version,expected_protocol='zero-shot')
        assert client.get_run(run).info.status=='FINISHED'
        receipt=dict(mode=mode,seed=seed,domain=domain,run_id=run,published=True)
        (target/'publication.json').write_text(json.dumps(receipt,indent=2));client.log_artifact(run,str(target/'publication.json'),'report')
        client.log_artifact(run,str(REPORT/'selection.json'),'study')
        publications.append(receipt)
        if domain=='private' and mode==selection['mode'] and (mode=='frozen' or seed==selection['seed']):selected_run=run;selected_folder=folder
    del pipeline,encoder;gc.collect();torch.cuda.empty_cache()
(REPORT/'publication.json').write_text(json.dumps(publications,indent=2))
from pettrace.evaluation.paired import paired_comparison
analysis={}
for domain in ('public','private'):
    baseline=json.loads((REPORT/'frozen'/domain/'evaluation-input.json').read_text())['result']
    analysis[domain]={}
    for mode in MODES:
        evaluations=[json.loads((REPORT/f'{mode}-{seed}'/domain/'evaluation-input.json').read_text())['result'] for seed in SEEDS]
        analysis[domain][mode]=dict(metrics={k:dict(mean=float(np.mean([e['test']['global'][k] for e in evaluations])),sd_seeds=float(np.std([e['test']['global'][k] for e in evaluations],ddof=1))) for k in ('rank1','rank2','rank5','mAP')},paired_vs_frozen={str(seed):paired_comparison(baseline,e) for seed,e in zip(SEEDS,evaluations)})
(REPORT/'comparison.json').write_text(json.dumps(analysis,indent=2))
for receipt in publications:client.log_artifact(receipt['run_id'],str(REPORT/'comparison.json'),'study')
if REGISTER_MODEL:
    bundle=selected_folder/'research-model';bundle.mkdir(exist_ok=True)
    for name in ('adapter.pt','metric.pt','golden.npy'):
        if (selected_folder/name).exists():shutil.copy2(selected_folder/name,bundle/name)
    (bundle/'model.json').write_text(json.dumps(dict(selection=selection,repository=REPOSITORY,revision=REVISION,processor=processor.to_dict(),detector_uri='models:/pettrace-embedder/2',base_hashes=base_hashes,serving_compatible=False),indent=2))
    import inspect
    import pettrace.training.dinov2 as implementation
    for module in (implementation,dino_metric,lora):
        shutil.copy2(inspect.getfile(module),bundle/Path(inspect.getfile(module)).name)
    client.log_artifacts(selected_run,str(bundle),'research-model')
    from pettrace.experiments.dinov2 import register_research_bundle
    register_research_bundle(REPORT,selected_run,client)
print('All evaluations published; registration is separate; no serving alias promoted.')
